# 01 — Research Question, Scope & Evaluation Protocol

**CuSelect-AI · Chapter I — From software readiness to scientific inquiry**

**Where we are coming from:** Notebook 00 verified that the environment can reproduce two initial *geometric* models. Those models say nothing yet about electrocatalytic performance.

**Question for this notebook:** *What claim are we trying to test, what would count as failure, and what evidence will we need?*

**Inputs:** `configs/CU-001_protocol.json` and `data/papers.csv`.

**Output:** `outputs/reports/01_protocol_check.json` and an approved study description for Notebook 02.


## 1.1 — Separate the chemical problem from the AI research problem

**Chemical domain:** selective electrochemical conversion of CO2 to ethylene on Cu-based catalytic surfaces.

**AI research task:** improve the **reliability of advancing or rejecting proposed catalysts and mechanistic claims**. We are *not* trying to claim a new ethylene Faradaic efficiency (FE) from an unrelaxed structure.

Initial materials: Cu(100) reference and one top-layer Ag substitution model. Because isolated substitutional Ag, Ag nanoparticles and Ag adatoms are distinct chemical configurations, later literature extraction must not conflate them.


In [5]:

# PSEUDOCODE
# 1. Find the parent directory of the notebooks folder.
# 2. Verify that it is the CuSelect-AI project root.
# 3. Load our scientific research protocol.
# 4. Display the campaign information.

from pathlib import Path
from cuselect.project_checks import require_project_root
from cuselect.protocol import load_research_protocol

# Locate and verify the project root
ROOT = require_project_root(Path.cwd().parent)

# Load the scientific research protocol
study = load_research_protocol(
    ROOT / "configs" / "CU-001_protocol.json"
)

# Display the research information
print("Project root:", ROOT.name)
print("Campaign:", study.campaign_id)
print("Research target:", study.system.target_product)
print("Initial candidates:", ", ".join(study.system.initial_candidates))
print("Predicted ethylene FE:", study.known_model_boundaries.selectivity_predicted)


Project root: CuSelect-AI
Campaign: CU-001
Research target: ethylene (C2H4)
Initial candidates: CU-001-REF, CU-001-AG1
Predicted ethylene FE: False


## 1.2 — Define a falsifiable hypothesis

**Working hypothesis (to be tested):** a mechanism-aware, evidence-grounded scientific critic can reduce unsupported advancement of Cu–Ag catalyst proposals compared with simpler ML or LLM screening, **without rejecting an unacceptable proportion of valid candidates**.

**Null hypothesis:** no reproducible improvement over a well-controlled baseline.

**Critical risk:** a model that rejects everything may appear to avoid false positives. Our evaluation therefore measures false rejections and abstentions alongside false advancement and cost. Thresholds and label adjudication rules must be fixed before final testing.


In [6]:
# PSEUDOCODE
# 1. Show the pre-registered metrics and control groups.
# 2. Confirm we have not accidentally declared the benchmark completed.

print('Primary metric:', study.evaluation.primary_metric)
print('Secondary metrics:', ', '.join(study.evaluation.secondary_metrics))
print('Controls:', ', '.join(study.evaluation.controls))
print('Holdout unit:', study.evaluation.holdout_group)
assert study.evaluation.evaluation_status == 'planned_not_run'


Primary metric: false_advancement_rate_at_defined_coverage
Secondary metrics: false_rejection_rate, abstention_rate, citation_support_rate, cost_per_validated_decision
Controls: descriptor_only, single_agent, multi_agent_no_critic
Holdout unit: DOI/publication (not random row)


## 1.3 — Define which claims would require additional evidence

| Claim | Minimum appropriate evidence |
|---|---|
| “This slab contains Ag in the top surface layer.” | Verified atomistic geometry and structure metadata |
| “This configuration has a favorable *CO adsorption descriptor.” | Comparable and validated energy calculations; explicit reference convention |
| “This catalyst favors ethylene over ethanol.” | Product-specific reaction-pathway or experimental evidence under stated conditions |
| “This catalyst is stable under CO2RR.” | Stability under relevant potentials, electrolytes and time scales |
| “CuSelect-AI improves scientific screening.” | DOI-held-out blind labels; appropriate baselines; errors, coverage and cost |

A predicted adsorption energy is not a direct estimate of ethylene FE. We do not generalize blindly across H-cells, flow cells and MEAs.


## 1.4 — Audit the **seed bibliography**, not the chemistry yet

The existing `data/papers.csv` contains candidate publications and bibliographic metadata collected during Stage 1. **It is a lead list, not a verified evidence corpus.** DOI/title matching, access permissions, relevant passages and exact claims still need to be checked in Notebook 02.


In [7]:
# PSEUDOCODE
# 1. Load bibliographic seed records only.
# 2. Validate that required columns exist.
# 3. Flag duplicate paper IDs or DOI entries for correction.
# 4. Do NOT infer measured catalyst performance from a paper's title.

import pandas as pd

papers = pd.read_csv(ROOT / 'data' / 'papers.csv')
required_columns = {'paper_id', 'year', 'title', 'doi', 'kind', 'why_relevant'}
assert required_columns.issubset(papers.columns)
assert papers['paper_id'].is_unique, 'Duplicate paper_id: resolve before Notebook 02'
assert papers['doi'].is_unique, 'Duplicate DOI: resolve before Notebook 02'
print('Bibliographic seed entries:', len(papers))
print('Bibliography review status: not yet fully verified')
papers[['paper_id', 'year', 'kind', 'doi']]


Bibliographic seed entries: 8
Bibliography review status: not yet fully verified


,paper_id,year,kind,doi
0,P001,2026,AI_and_Cu_CO2RR,10.1002/anie.202524612
1,P002,2026,multi_agent_DFT,10.1038/s41524-026-02139-1
2,P003,2026,multi_agent_MLFF,10.1021/acscentsci.6c01260
3,P004,2026,scientific_verification,10.1038/s44387-026-00111-4
4,P005,2024,Cu_Ag_experimental,10.1038/s41467-024-54636-w
5,P006,2024,scientific_dataset,10.1038/s41597-024-03180-9
6,P007,2026,dual_site_experiment,10.1002/anie.5598898
7,P008,2026,constant_potential_MLIP,10.1016/j.esci.2026.100642


## 1.5 — Protect the future validation set

When we build a literature agent, we must divide examples by publication or experiment family. A source paper used to define a blind test label must not be searchable by the AI during that test. A normal random split of rows from the same paper would risk leakage.

The first benchmark protocol is **planned**, not yet evaluated. Numerical thresholds and independent labels will be decided after inspecting available data, then frozen before final evaluation.


In [8]:
# PSEUDOCODE
# 1. Build a structured study readiness report.
# 2. Record the current limits and planned evaluation without inventing results.
# 3. Save the report for inspection and the next notebook.

import json

readiness = {
    'campaign_id': study.campaign_id,
    'protocol_schema_version': study.schema_version,
    'seed_papers_count': int(len(papers)),
    'seed_doi_count': int(papers['doi'].notna().sum()),
    'literature_status': 'bibliographic_seeds_not_claim_verified',
    'evaluation_status': study.evaluation.evaluation_status,
    'holdout_group': study.evaluation.holdout_group,
    'scientific_predictions_available': False,
    'next_notebook': '02_Literature_and_Evidence',
}

output = ROOT / 'outputs' / 'reports' / '01_protocol_check.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(readiness, indent=2) + '\n', encoding='utf-8')
print('Saved:', output.relative_to(ROOT))
readiness


Saved: outputs/reports/01_protocol_check.json


{'campaign_id': 'CU-001',
 'protocol_schema_version': '0.1',
 'seed_papers_count': 8,
 'seed_doi_count': 8,
 'literature_status': 'bibliographic_seeds_not_claim_verified',
 'evaluation_status': 'planned_not_run',
 'holdout_group': 'DOI/publication (not random row)',
 'scientific_predictions_available': False,
 'next_notebook': '02_Literature_and_Evidence'}

## 1.6 — Interpretation and handoff → Notebook 02

**What we have established:** a falsifiable question, explicit candidate scope, evaluation metrics, control groups, bibliographic leads and a DOI-grouped holdout policy.

**What we do not know:** whether the scientific papers are accurately represented in the seed register; whether matching computational and experimental observations exist; what level of model error we can achieve. Those are empirical questions.

**For Notebook 02:** verify papers, record metadata/DOI/license, extract *specific* mechanistic claims with exact source locations, distinguish reported observations from hypotheses, and preserve contradictions. Produce a source-grounded evidence table that can be audited independently. Do **not** silently fill missing values.
